# Friend A — L5 DaughterNet-B (full resolution, resumable)

This is the current L5 task from the top block of `PROMPT_FRIEND_A.md`. It trains five models from random
initialisation on the registered leak-safe folds and returns honest held-out-fold probabilities for all 1,012,255
candidate nodes.

Fixed contract: full XY resolution, crop `16×64×64`, image lags `−2…+3`, classes `cont / parent / daughter`, RTX GPU.
The labels, candidate-pool hard examples, optimiser and per-source OOF readout use the current L2 v2 trainer.

Run the cells in order. The full run is resumable. The first build creates a roughly **25 GiB** cache outside Git.
Interrupting during the build keeps completed videos; interrupting during training keeps the last completed epoch;
scoring keeps completed videos. Rerun the same long cell to continue. Do not delete the `active` output directory.

The final hand-off is `experiments/local_friend_a_l5_daughternet_b/oof_nodes.parquet` with exactly
`stem,node_id,fold,p_cont,p_par,p_dau`. No leaderboard submission is made here.

Conservative local RTX 5070 Ti estimate for the current v2 sample set: **14–22 hours total** (cache 1–2 h, training
12–18 h, final million-node scoring 1–2 h). Early stopping may shorten training; actual epoch timings are written.


In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start, *start.parents, start / ".github-push-worktree"]
    for candidate in candidates:
        if (candidate / "PROMPT_FRIEND_A.md").is_file() and (
            candidate / "experiments" / "local_friend_a_l5_daughternet_b" / "runner.py"
        ).is_file():
            return candidate
    raise FileNotFoundError("Repository checkout not found. Start Jupyter from C:\\Projects\\CellTracker.")

PROJECT_ROOT = find_repo_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiments.local_friend_a_l5_daughternet_b.runner import (
    L5Config, clear_scoring_stop, clear_training_stop, compact_scoring_status,
    compact_training_status, preflight, request_scoring_stop, request_training_stop,
    run_node_scoring, run_smoke, run_training,
)

cfg = L5Config.from_environment(PROJECT_ROOT)
print("Repository:", PROJECT_ROOT)
print("Data:", cfg.data_dir)
print("Working outputs:", cfg.work_dir)
print("Final OOF:", cfg.final_oof_path)
cfg


## 1. Mandatory preflight

Checks registered inputs, all 199 videos, CUDA, model shape, and immutable hashes.

In [ ]:
preflight_report = preflight(cfg)
preflight_report

## 2. End-to-end smoke

Six division-rich videos, two temporary folds, one epoch. Full training must not start if this fails.

In [ ]:
clear_training_stop(cfg, smoke=True)
smoke_status = run_smoke(cfg)
smoke_status

## 3. Full five-fold training

This is the first long cell. On the first run it builds the ~25 GiB v2 crop cache, then trains one fold at a time on the
RTX 5070 Ti. Rerunning it reuses the cache, skips finished folds, and resumes an unfinished fold from `last.pt`.

To stop immediately, interrupt the kernel; the current in-memory part is lost, but completed cache videos / the last
completed epoch remain. For a clean stop after the current epoch, create the displayed `STOP_REQUESTED` file from a
second PowerShell or a second notebook kernel. Do not execute the helper below before starting unless you intend to stop.


In [ ]:
clear_training_stop(cfg)
training_status = run_training(cfg)
training_status

### Training status (safe to run after an interrupt)

In [ ]:
compact_training_status(cfg)

### Optional graceful-stop helper

Run this from a **second kernel** while training is active. It asks the worker to finish the current epoch, save
`last.pt`, and stop before another fold. The normal training cell clears this marker when you deliberately resume.


In [ ]:
# request_training_stop(cfg)  # uncomment only when you want to stop
# compact_training_status(cfg)

## 4. Score all exported candidate nodes with the held-out model

Run only after all five fold summaries exist. Each video uses the model of its registered fold (the model that did not
see that video). Results are checkpointed per video, so rerunning continues where it stopped.


In [ ]:
clear_scoring_stop(cfg)
scoring_report = run_node_scoring(cfg)
scoring_report

### Scoring status / optional graceful stop

Use `request_scoring_stop(cfg)` from a second kernel to stop after the current video.

In [ ]:
# request_scoring_stop(cfg)  # uncomment only when you want to stop
compact_scoring_status(cfg)

## Hand-off

When scoring reports `status: complete`, commit only the final
`experiments/local_friend_a_l5_daughternet_b/oof_nodes.parquet` plus a short results note. Never commit the 25 GiB cache,
fold checkpoints, per-video parquet files, or raw data. Send Vitalii the commit hash and confirm the exact row count,
schema, and SHA-256 printed by the scoring report.
